# Chapter 6 &mdash; Which DFAs a $k$-Window Can Learn

**Concept 14 of the Chapter 6 decomposition:** *Which DFAs a $k$-Window Can Learn: Myhill-Nerode Against a Window*

Decidable without training anything &mdash; and the smallest machine in the table is the impossible one.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter6-DFAOps/Concept-Which-DFAs-A-Window-Can-Learn/Concept-Which-DFAs-A-Window-Can-Learn.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
# Run me first.  Works on Colab and on a local Jove checkout.
import os, subprocess, sys

def _git(*a):
    r = subprocess.run(('git',) + a, capture_output=True, text=True)
    return r.stdout.strip() if r.returncode == 0 else ''

REPO = 'https://github.com/ganeshutah/Jove'
try:                       # ---- Colab: clone once, pull thereafter ----
    import google.colab
    was = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD') if os.path.isdir('Jove') else ''
    if os.path.isdir('Jove') and not was:
        print('Jove: WARNING ./Jove exists but is not a git checkout -- left as is')
    elif was:
        _git('-C', 'Jove', 'pull', '-q', '--ff-only')
        now = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD')
        if now and now != was:
            print('Jove: PULLED  %s -> %s' % (was, now))
            print(_git('-C', 'Jove', 'log', '--oneline', was + '..' + now))
        else:
            print('Jove: PULLED  already current at %s' % (now or was))
    else:
        _git('clone', '-q', REPO, 'Jove')
        print('Jove: CLONED  at %s' % (_git('-C', 'Jove', 'rev-parse',
                                             '--short', 'HEAD') or '?'))
    JOVE = 'Jove'
except ImportError:        # ---- local: the checkout above Chapter<N>/ ----
    JOVE = next((p for p in ('../..', '../../..', '..', '.')
                 if os.path.isdir(os.path.join(p, 'jove'))), '../..')
    print('Jove: LOCAL   checkout at %s'
          % (_git('-C', JOVE, 'rev-parse', '--short', 'HEAD') or '?'))
sys.path.insert(0, JOVE)

# A session can already hold an OLDER jove in sys.modules.  The pull above
# updates the files on disk, but `import` would hand back the cached module --
# so a fixed library still behaves like the broken one.  Drop them first.
for _m in [k for k in list(sys.modules) if k == 'jove' or k.startswith('jove.')]:
    del sys.modules[_m]

from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
from jove.GPTLab         import *

import jove; print('Jove loaded from', list(jove.__path__)[0])

## 1. The idea


The previous concept trained a model and compared it with a DFA. This one answers the
same question **without training anything**, which is the more useful skill.

The model's state is the last $k$ symbols. So if two prefixes share their last $k$
symbols but land in **different states of the minimal DFA**, those states are
distinguishable &mdash; Concept 6 &mdash; while the model sees the same input in both
cases. It must be wrong on one of them, for **any** weights, on **any** corpus.

That is a decision procedure. `k_local(D, k)` checks whether every $k$-window is
reachable into exactly one state of the minimal machine. It is Myhill&ndash;Nerode
(Concept 7) run against a window instead of against all of history.

**And the answer has nothing to do with size**, which is the surprise worth keeping.

## 2. Definitions

### Four languages

In [ ]:
# --- the languages we will ask about ------------------------------------
ENDS01 = md2mc('''DFA
I  : 0 -> S0
I  : 1 -> I
S0 : 0 -> S0
S0 : 1 -> F
F  : 0 -> S0
F  : 1 -> I
''')

PARITY = md2mc('''DFA
IF : 0 -> IF
IF : 1 -> S1
S1 : 0 -> S1
S1 : 1 -> IF
''')

DIV3 = md2mc('''DFA
IF : 0 -> IF
IF : 1 -> S1
S1 : 0 -> S2
S1 : 1 -> IF
S2 : 0 -> S1
S2 : 1 -> S2
''')

NO11 = md2mc('''DFA
IF : 0 -> IF
IF : 1 -> F1
F1 : 0 -> IF
F1 : 1 -> D
D  : 0|1 -> D
''')

for nm, D in (('ends in 01', ENDS01), ('even # of 1s', PARITY),
              ('value div by 3', DIV3), ('no two 1s in a row', NO11)):
    print('  %-22s minimal DFA: %d states' % (nm, len(min_dfa(D)['Q'])))

<!-- nav-strip -->

---

&larr;&nbsp;[Ch6&nbsp;13.&nbsp;A Transformer with Context $k$ is a Finite-State Machine](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter6-DFAOps/Concept-Transformer-Is-Finite-State/Concept-Transformer-Is-Finite-State.ipynb) &nbsp;&middot;&nbsp; [**Chapter 6** index](https://github.com/ganeshutah/Jove/blob/master/Chapter6-DFAOps/README.md) &nbsp;&middot;&nbsp; [Ch6&nbsp;15.&nbsp;Watching It Fail: Parity, and What Hedging Looks Like](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter6-DFAOps/Concept-Watching-It-Fail-Parity/Concept-Watching-It-Fail-Parity.ipynb)&nbsp;&rarr;

---

## 3. Tests

**The test**, for each language and each window size.

In [ ]:
from jove.GPTLab import k_local, colliding_prefixes

print('%-22s %8s   %s' % ('language', 'min-DFA', 'learnable at k = 1..6?'))
for nm, D in (('ends in 01', ENDS01), ('no two 1s in a row', NO11),
              ('even # of 1s', PARITY), ('value div by 3', DIV3)):
    marks = ' '.join('%d:%s' % (k, 'Y' if k_local(D, k)[0] else '.')
                     for k in range(1, 7))
    print('%-22s %5d    %s' % (nm, len(min_dfa(D)['Q']), marks))

**Read the table again.** The two-state machine is the one that never becomes learnable.

In [ ]:
print('ends in 01          3 states   learnable from k = 2')
print('no two 1s in a row  3 states   learnable from k = 1')
print('even # of 1s        2 states   NEVER')
print('value div by 3      3 states   NEVER')
print()
print('Size is not the obstacle.  A window of any length can be told')
print('everything about the last k symbols and nothing about how many')
print('symbols came before -- and parity is a fact about the count.')
assert k_local(PARITY, 6)[0] is False
assert k_local(ENDS01, 2)[0] is True

**The counterexample, concretely.** Two strings the model cannot tell apart that the language must separate.

In [ ]:
for nm, D in (('even # of 1s', PARITY), ('value div by 3', DIV3)):
    a, b, tail = colliding_prefixes(D, 3)
    print('%s:' % nm)
    print('   %-6s and %-6s both end in %r' % (repr(a), repr(b), tail))
    print('   accepted? %-6s %s' % (accepts_dfa(D, a), accepts_dfa(D, b)))
    assert accepts_dfa(D, a) != accepts_dfa(D, b)
    print()
print('Same last three symbols, so the same model state -- and the language')
print('demands opposite answers.  No corpus repairs that.')

Why `no two 1s in a row` **is** learnable, at $k=1$.

In [ ]:
print('The forbidden thing is local: a 1 followed by a 1.')
print('One symbol of memory is enough to refuse the second.')
print()
ok, w, land = k_local(NO11, 1)
print('k_local(NO11, 1) ->', ok)
print()
print('That is the shape of every learnable case: the constraint is about a')
print('bounded window, so a bounded window can enforce it.')
print()
print('A caution worth carrying into the next concept.  For THIS language')
print('every window that ever occurs is accepted -- 11 cannot appear inside')
print('an accepted string at all -- so P(END) has nothing to separate.  The')
print('language constrains what may FOLLOW, not where you may stop, and the')
print('probe for it is P(1 | ...1), which the model drives to zero.')

## 4. Exercises


1. Build the DFA for "contains `010`". Is it $k$-local, and at which $k$?
2. "Even number of 1s" fails. Does "even **length**" fail too? Predict, then test.
3. Prove the criterion: show that if every $k$-window lands in one state of the
   minimal DFA, then a table from windows to next-symbol behaviour exists.
4. The criterion uses the **minimal** DFA. Show it would be wrong on a non-minimal
   one, by padding a machine with a duplicate state.
5. Languages passing this test for some $k$ are called *strictly locally testable*.
   Where do they sit relative to the regular languages, and what does the table
   above demonstrate about that?

In [ ]:
# Your work for the exercises above.

## 5. Where next

In [ ]:
# Previous / next, and a search box for every concept.
# Type a chapter (Chapter7, ch7, NFA) or words from a title (pumping).
#
# Following a link opens a NEW Colab runtime. To pull another concept's
# definitions into THIS session instead:
#     load_here('Chapter7-NFA/Concept-...')
import os, sys
try:                       # usually already done by the Setup cell
    import jove
except ModuleNotFoundError:
    _p = next((p for p in ('Jove', '../..', '../../..', '..', '.')
               if os.path.isdir(os.path.join(p, 'jove'))), None)
    if _p:
        sys.path.insert(0, _p)
try:
    from jove.Nav import nav, load_here
    nav(here='Chapter6-DFAOps/Concept-Which-DFAs-A-Window-Can-Learn')
except ModuleNotFoundError:
    print('Jove is not on the path yet.')
    print('Run the Setup cell at the top of this notebook, then re-run this one.')